In [1]:
import findspark
import os
#findspark.init() 
SPARK_HOME='/opt/cloudera/parcels/CDH/lib/spark'
# SPARK_HOME='/home/qiany/.conda/envs/py37'
# os.environ['SPARK_HOME'] = '/home/qiany/.conda/envs/py37'
findspark.init(SPARK_HOME)

In [2]:
import time
import math
import copy
import csv
import json
import os
import codecs
import subprocess
#from hdfs import InsecureClient
import numpy as np
#from pyspark import SparkContext
from pyspark import SQLContext
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.functions import expr, create_map, array_union,flatten,array_sort,coalesce,broadcast,collect_list, collect_set, udf, array_remove, log, lit, first, col, array, sort_array,split, explode, desc, asc, row_number,isnan, when, count
from pyspark.sql.types import *
import rtree
from pyspark.sql import Window
#import igraph
#from igraph import Graph
import geofeather
from pyspark.storagelevel import StorageLevel

In [3]:
# /local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410_final/ALS_L1B_20190410T174554_181213_small_crop_as_100000.off
# /local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410_final/ALS_L1B_20190410T174554_181213_3_as_100000.off
# /local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410_final/ALS_L1B_20190410T174554_181213_small_crop_tiny_test_as_100000.off
# /local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410_final/ALS_L1B_20190410T174554_181213_3_crop_as_100000.off
# /local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410T183726_185444_1/ALS_L1B_20190410T183726_185444_1.off
# /local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410T153919_155000_1/ALS_L1B_20190410T153919_155000_1.off
# /local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410T174554_181213/ALS_L1B_20190410T174554_181213.off

t_whole_program_0 = time.time()

tin_file = input("Here is a programe to compute the Forman gradient, please input the absolute or relative path in local disk to your TIN file:")
# tin_file = '/local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410_final/ALS_L1B_20190410T174554_181213_small_crop_as_100000.off'

print("\n********************")

# get the directory to the TIN file
tin_directory = os.path.dirname(tin_file)
print("tin_directory: ", tin_directory)

# seaice
# seaice_11122025
# seaice_simplify_ele
# ALS_L1B_20190410T174554_181213_small_crop_tiny_test
# ALS_L1B_20190410T174554_181213_small_crop_tiny_test_pers_0.5
# ALS_L1B_20190410T174554_181213_3_crop_pers_0.25
# ALS_L1B_20190410T174554_181213_3_pers_025
# ALS_L1B_20190410T174554_181213_3_pers_050
# ALS_L1B_20190410T174554_181213_3_pers_010
# ALS_L1B_20190410T174554_181213_3_pers_020
# ALS_L1B_20190410T174554_181213_3_pers_040
# ALS_L1B_20190410T174554_181213_3_pers_030
# ALS_L1B_20190410T183726_185444_1_pers_025
# ALS_L1B_20190410T153919_155000_1_pers_025
# ALS_L1B_20190410T174554_181213_pers_025
# ALS_L1B_20190410T174554_181213_3_pers_015
# ALS_L1B_20190410T174554_181213_3_pers_035
# ALS_L1B_20190410T174554_181213_3_pers_045

# ALS_L1B_20190410T153919_155000_1_pers_020
# ALS_L1B_20190410T153919_155000_1_pers_015
# ALS_L1B_20190410T153919_155000_1_pers_010
# ALS_L1B_20190410T153919_155000_1_pers_030
# ALS_L1B_20190410T153919_155000_1_pers_035
# ALS_L1B_20190410T153919_155000_1_pers_040
# ALS_L1B_20190410T153919_155000_1_pers_045
# ALS_L1B_20190410T153919_155000_1_pers_050

# ALS_L1B_20190410T183726_185444_1_pers_020
# ALS_L1B_20190410T183726_185444_1_pers_015
# ALS_L1B_20190410T183726_185444_1_pers_010
# ALS_L1B_20190410T183726_185444_1_pers_030
# ALS_L1B_20190410T183726_185444_1_pers_035
# ALS_L1B_20190410T183726_185444_1_pers_040
# ALS_L1B_20190410T183726_185444_1_pers_045
# ALS_L1B_20190410T183726_185444_1_pers_050

directory = input("The HDFS directory where you want to store DataFrames:")

# simplify_with_order = input("Simplify with order('y' or 'n'):") or "n"
simplify_with_order = "n"

porsist_thre = input("What is the persistence value threshold (m) you want to set:") or "0.25"

already_resimplify_idx = input("How many resimplification have you already performed:") or "1"

round_idx = input("which round of merging you want to perform:") or "2"

# get the basename to the TIN file
tin_basename = os.path.basename(tin_file) # input_vertices_2.off
print("tin_basename: ", tin_basename)

# get the filename of the TIN file
tin_filename = os.path.splitext(tin_basename)[0] # input_vertices_2
print("tin_filename: ", tin_filename)

# get the type of TIN file: off, tri, etc
tin_extension = os.path.splitext(tin_basename)[1] # .off
print("tin_extension: ", tin_extension)
print("\n********************")

Here is a programe to compute the Forman gradient, please input the absolute or relative path in local disk to your TIN file: /local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410T183726_185444_1/ALS_L1B_20190410T183726_185444_1.off



********************
tin_directory:  /local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410T183726_185444_1


The HDFS directory where you want to store DataFrames: ALS_L1B_20190410T183726_185444_1_pers_050
What is the persistence value threshold (m) you want to set: 0.50
How many resimplification have you already performed: 1
which round of merging you want to perform: 2


tin_basename:  ALS_L1B_20190410T183726_185444_1.off
tin_filename:  ALS_L1B_20190410T183726_185444_1
tin_extension:  .off

********************


In [4]:
# filtra is the order of each vertex, the order is obtained by ranking elevation values of vertices
# filtra = input("Do you have filtration data?") or "yes"
filtra = 'yes'

if filtra.lower() == 'no':    
    Basic_Data = input("Do you have basic pts and tri data?")
    
Num_executor = input("spark.executor.instances:") or "32"
Num_core_per_executor = input("spark.executor.cores:") or "3"
Memory_executor = input("spark.executor.memory? Please end with 'g':") or "56g"
MemoryOverhead_executor = input("spark.executor.memoryOverhead? Please end with 'g':") or "24g"

# Num_core_per_driver = Num_core_per_executor
# Memory_driver = Memory_executor
# MemoryOverhead_driver = MemoryOverhead_executor

Num_core_per_driver = '5'
Memory_driver = '64g'
MemoryOverhead_driver = '32g'

Num_shuffle_partitions = input("spark.sql.shuffle.partitions:") or "288"

spark.executor.instances: 
spark.executor.cores: 
spark.executor.memory? Please end with 'g': 
spark.executor.memoryOverhead? Please end with 'g': 
spark.sql.shuffle.partitions: 


In [5]:
from pyspark.sql import SparkSession
from pyspark import StorageLevel
import geopandas as gpd
import pandas as pd
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, FloatType, ArrayType, MapType
from shapely.geometry import Point
from shapely.geometry import Polygon

from sedona.register import SedonaRegistrator
from sedona.core.SpatialRDD import SpatialRDD, PointRDD, CircleRDD, PolygonRDD, LineStringRDD
from sedona.core.enums import FileDataSplitter
from sedona.utils.adapter import Adapter
from sedona.core.spatialOperator import KNNQuery
from sedona.core.spatialOperator import JoinQuery
from sedona.core.spatialOperator import JoinQueryRaw
from sedona.core.spatialOperator import RangeQuery
from sedona.core.spatialOperator import RangeQueryRaw
from sedona.core.formatMapper.shapefileParser import ShapefileReader
from sedona.core.formatMapper import WkbReader
from sedona.core.formatMapper import WktReader
from sedona.core.formatMapper import GeoJsonReader
from sedona.sql.types import GeometryType
from sedona.core.enums import GridType
from sedona.core.SpatialRDD import RectangleRDD
from sedona.core.enums import IndexType
from sedona.core.geom.envelope import Envelope
from sedona.utils import SedonaKryoRegistrator, KryoSerializer

In [6]:
os.environ['PYSPARK_PYTHON'] = "./environment/bin/python"
#os.environ['PYSPARK_PYTHON'] = "/home/qiany/.conda/envs/py37/bin/python"
os.environ['YARN_CONF_DIR'] = "/opt/cloudera/parcels/CDH/lib/spark/conf/yarn-conf"

In [7]:
'''
spark.executor.cores: # Number of concurrent tasks an executor can run, euqals to the number of cores to use on each executor
spark.executor.instances: # Number of executors for the spark application
spark.executor.memory: # Amount of memory to use for each executor that runs the task
spark.executor.memoryOverhead:
spark.driver.cores: # Number of cores to use for the driver process; the default number is 1
spark.driver.memory: # Amount of memory to use for the driver
spark.driver.maxResultSize: to define the maximum limit of the total size of the serialized result that a driver can store for each Spark collect action
spark.default.parallelism: # Default number of partitions in RDDs returned by transformations like join, reduceByKey, and parallelize when not set by user. It can be set as spark.executor.instances * spark.executor.cores * 2
spark.sql.shuffle.partitions: determine how many partitions are used when data is shuffled between nodes, e.g., joins or aggregations. usually 1~5 times of executor.instances * executor.cores
spark.memory.storageFraction: determines the fraction of the heap space that is allocated to caching RDDs and DataFrames in memory.
spark.kryoserializer.buffer.max: determine the maximum of data that can be serialized at once; this must be larger than any object we attempt to serialize
spark.rpc.message.maxSize: # Maximum message size (in MiB) to allow in "control plane" communication; generally only applies to map output size information sent between executors and the driver. To communicate between the nodes, Spark uses a protocol called RPC (Remote Procedure Call), which sends messages back and forth. The spark.rpc.message.maxSize parameter limits how big these messages can be. 
spark.sql.broadcastTimeout: Spark will wait for this amount of time before giving up on broadcasting a table. Broadcasting can take a long time if the table is large or if there is a shuffle operation before it.
spark.sql.autoBroadcastJoinThreshold: Spark will broadcast a table to all worker nodes when performing a join if its size is less than this value; -1 means disabling broadcasting
'''

date = time.strftime("%m,%d,%Y")
date_name = date.split(',')[0] + date.split(',')[1] + date.split(',')[2]

hour = time.strftime("%H,%M")
hour_name = hour.split(',')[0] + hour.split(',')[1]

# set the Spark app name
spark_app_name = "Seaice_step8_merge_r" + round_idx + "_step2_save_graph_and_result_con_" + tin_filename + '_' + date_name + '_' + hour_name
print("spark_app_name:", spark_app_name)

spark = SparkSession \
.builder \
.appName(spark_app_name) \
.master('yarn') \
.config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
.config("spark.kryo.registrator", SedonaKryoRegistrator.getName) \
.config('spark.jars','sedona-core-2.4_2.11-1.0.0-incubating.jar,sedona-sql-2.4_2.11-1.0.0-incubating.jar,sedona-python-adapter-2.4_2.11-1.0.0-incubating.jar,sedona-viz-2.4_2.11-1.0.0-incubating.jar,geotools-wrapper-geotools-24.0.jar,graphframes-0.8.0-spark2.4-s_2.11.jar') \
.config('spark.executor.cores', Num_core_per_executor) \
.config('spark.executor.instances', Num_executor) \
.config('spark.executor.memory', Memory_executor) \
.config('spark.executor.memoryOverhead', MemoryOverhead_executor) \
.config('spark.driver.cores', Num_core_per_driver) \
.config('spark.driver.memory', Memory_driver) \
.config('spark.driver.memoryOverhead', MemoryOverhead_driver) \
.config('spark.driver.maxResultSize', '0') \
.config('spark.dynamicAllocation.enabled', 'false') \
.config('spark.network.timeout', '10000001s') \
.config('spark.executor.heartbeatInterval', '10000000s') \
.config('spark.sql.shuffle.partitions', Num_shuffle_partitions) \
.config("spark.default.parallelism", '200') \
.config("spark.kryoserializer.buffer.max", "1024mb") \
.config('spark.rpc.message.maxSize', '256') \
.config("spark.sql.broadcastTimeout", "36000") \
.config("spark.sql.autoBroadcastJoinThreshold", "-1") \
.config("spark.python.profile", "true") \
.config("spark.eventLog.enabled", "true") \
.config('spark.yarn.dist.archives', '/local/data/yuehui/py37.tar.gz#environment') \
.getOrCreate()

spark_app_name: Seaice_step8_merge_r2_step2_save_graph_and_result_con_ALS_L1B_20190410T183726_185444_1_04092026_1512


In [8]:
import sys
sys.path.append("/local/data/yuehui/pyspark/Topo_Simplification/code")

In [9]:
import graphframes
from graphframes import GraphFrame
from graphframes import *
from graphframes.lib import Pregel

In [10]:
file_df_ver_order = directory + '/' + tin_filename + '_filtra_pts' + '.parquet'
df_ver_order = spark.read.parquet(file_df_ver_order)
df_ver_order.printSchema()

root
 |-- x: float (nullable = true)
 |-- y: float (nullable = true)
 |-- ele: float (nullable = true)
 |-- self_index: integer (nullable = true)
 |-- self_order: integer (nullable = true)



### Graph_VE

In [11]:
if round_idx == '1':
    file_df_V1_result_VE_update_final = directory + '/' + tin_filename + '_df_V1_result_VE_update_eliminate_resimplify_r' + already_resimplify_idx + '.parquet'
else:
    file_df_V1_result_VE_update_final = directory + '/' + tin_filename + '_df_V1_result_VE_update_eliminate_from_r' + already_resimplify_idx + '_merge_r' + str(int(round_idx)-1) + '.parquet'

# file_df_V1_result_VE_update_final = directory + '/' + tin_filename + '_df_V1_result_VE_update_eliminate_resimplify_r1' + '.parquet'
df_V1_result_VE_update_final = spark.read.parquet(file_df_V1_result_VE_update_final)
df_V1_result_VE_update_final.printSchema()

root
 |-- component: long (nullable = true)
 |-- final_V1_paths: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)



In [12]:
df_V1_result_VE_update_final_expl = df_V1_result_VE_update_final.select(explode(df_V1_result_VE_update_final.final_V1_paths).alias("V1_path"))
df_V1_result_VE_update_final_expl.printSchema()

root
 |-- V1_path: array (nullable = true)
 |    |-- element: integer (containsNull = true)



#### create an arc DataFrame from V1-paths for the updated version

In [13]:
df_graph_V1_arc_resimplify_r1 = (
    df_V1_result_VE_update_final_expl
    # head = all except last
    .withColumn("head", F.expr("slice(V1_path, 1, size(V1_path)-1)"))
    # tail = all except first
    .withColumn("tail", F.expr("slice(V1_path, 2, size(V1_path)-1)"))
    # pair up head[i], tail[i]
    .withColumn("pairs", F.arrays_zip("head", "tail"))
    # one row per (src, dst)
    .select(F.explode("pairs").alias("e"))
    .select(F.col("e.head").alias("src"), F.col("e.tail").alias("dst"))
)

df_graph_V1_arc_resimplify_r1.printSchema()

root
 |-- src: integer (nullable = true)
 |-- dst: integer (nullable = true)



In [14]:
# drop duplicates
df_graph_V1_arc_resimplify_r1 = df_graph_V1_arc_resimplify_r1.dropDuplicates(["src", "dst"])
df_graph_V1_arc_resimplify_r1.printSchema()

root
 |-- src: integer (nullable = true)
 |-- dst: integer (nullable = true)



In [15]:
t0 = time.time()

file_df_graph_V1_arc_resimplify_r1 = directory + '/' + tin_filename + '_VE_arc_from_r' + already_resimplify_idx + '_merge_r' + round_idx + '.parquet'

# file_df_graph_V1_arc_resimplify_r1 = directory + '/' + tin_filename + '_VE_arc_from_r1_merge_r1' + '.parquet'
df_graph_V1_arc_resimplify_r1.write.parquet(file_df_graph_V1_arc_resimplify_r1) 

t1 = time.time()
print("time cost:", t1-t0)

time cost: 13.220040082931519


#### the nodes are the same as that of before simplification

In [16]:
file_df_graph_V1_node = directory + '/' + tin_filename + '_VE_node' + '.parquet'
df_graph_V1_node = spark.read.parquet(file_df_graph_V1_node)
df_graph_V1_node.printSchema()

root
 |-- id: integer (nullable = true)



#### construct a Graph_VE

In [17]:
graph_VE = GraphFrame(df_graph_V1_node, df_graph_V1_arc_resimplify_r1)

In [18]:
# set a checkpoint directory to improve performance
# Checkpointing regularly helps recover from failures, clean shuffle files, shorten the
# lineage of the computation graph, and reduce the complexity of plan optimization.

spark.sparkContext.setCheckpointDir('checkpoints')

In [19]:
t_con_0 = time.time()
# result_con consists of two columns, vertex id, component
result_con_VE = graph_VE.connectedComponents()
result_con_VE.printSchema()

t_con_1 = time.time()
t_con = t_con_1 - t_con_0
print("time cost of connected components:",t_con)

root
 |-- id: integer (nullable = true)
 |-- component: long (nullable = true)

time cost of connected components: 72.95531415939331


#### distinct number of components

In [20]:
result_con_VE_gpy = result_con_VE.groupby('component').agg(collect_list('id').alias('ids'))
result_con_VE_gpy.printSchema()

root
 |-- component: long (nullable = true)
 |-- ids: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [21]:
result_con_VE_gpy_filtered = (
    result_con_VE_gpy
    .filter(F.size(F.col("ids")) > 1)
)

num_comp_distinct = result_con_VE_gpy_filtered.count()
print("The number of distinct components:", num_comp_distinct)

The number of distinct components: 153791


In [22]:
# _df_crit_V_update_no_threshold is the critical vertices after update but without resimplification
if round_idx == '1':
    file_df_crit_V = directory + '/' + tin_filename + '_df_crit_V_update_resimplify_r' + already_resimplify_idx + '.parquet'
else:
    file_df_crit_V = directory + '/' + tin_filename + '_df_crit_V_update_from_r' + already_resimplify_idx + '_merge_r' + str(int(round_idx)-1) + '.parquet'

# file_df_crit_V = directory + '/' + tin_filename + '_df_crit_V_update_resimplify_r1' + '.parquet'
df_crit_V = spark.read.parquet(file_df_crit_V)
df_crit_V.printSchema()

num_df_crit_V = df_crit_V.count()
print("The number of critical vertices in df_crit_V_update:", num_df_crit_V)

root
 |-- Ver: integer (nullable = true)
 |-- Min_ver: integer (nullable = true)

The number of critical vertices in df_crit_V_update: 156976


In [23]:
result_con_VE_resimplify_r1_final = result_con_VE_gpy_filtered.select(explode("ids").alias("id"), col("component"))
result_con_VE_resimplify_r1_final.printSchema()

root
 |-- id: integer (nullable = true)
 |-- component: long (nullable = true)



In [24]:
t_0 = time.time()

file_result_con_VE_resimplify_r1_final = directory + '/' + tin_filename + '_result_con_VE_from_r' + already_resimplify_idx + '_merge_r' + round_idx + '.parquet'

# file_result_con_VE_resimplify_r1_final = directory + '/' + tin_filename + '_result_con_VE_from_r1_merge_r1' + '.parquet'
result_con_VE_resimplify_r1_final.write.parquet(file_result_con_VE_resimplify_r1_final)

t_1 = time.time()
print("time cost of connected components:",t_1 - t_0)

time cost of connected components: 2.7700068950653076


### Graph_ET

In [25]:
if round_idx == '1':
    file_df_V2_result_ET_update_final = directory + '/' + tin_filename + '_df_V2_result_ET_update_eliminate_resimplify_r' + already_resimplify_idx + '.parquet'
else:
    file_df_V2_result_ET_update_final = directory + '/' + tin_filename + '_df_V2_result_ET_update_from_r' + already_resimplify_idx + '_merge_r' + str(int(round_idx)-1) + '.parquet'

# file_df_V2_result_ET_update_final = directory + '/' + tin_filename + '_df_V2_result_ET_update_eliminate_resimplify_r1' + '.parquet'
df_V2_result_ET_update_final = spark.read.parquet(file_df_V2_result_ET_update_final)
df_V2_result_ET_update_final.printSchema()

root
 |-- component: long (nullable = true)
 |-- pot_maximum: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- pot_saddle: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- two_Saddle_Co_t_id: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- V2_paths: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- multi_Saddles_co_tri_id: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- reversed_V2_paths: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)



In [26]:
if round_idx == '1':
    df_V2_result_ET_update_final_expl = df_V2_result_ET_update_final.select(explode(df_V2_result_ET_update_final.final_V2_paths).alias("V2_path"))
else:
    df_V2_result_ET_update_final_expl = df_V2_result_ET_update_final.select(explode(df_V2_result_ET_update_final.reversed_V2_paths).alias("V2_path")).filter(F.col("V2_path").isNotNull()).filter(F.size("V2_path") >= 2)
    
df_V2_result_ET_update_final_expl.printSchema()

root
 |-- V2_path: array (nullable = true)
 |    |-- element: integer (containsNull = true)



#### create an arc DataFrame from V2-paths for the updated version

In [27]:
df_graph_V2_arc_resimplify_r1 = (
    df_V2_result_ET_update_final_expl
    # head = all except last
    .withColumn("head", F.expr("slice(V2_path, 1, size(V2_path)-1)"))
    # tail = all except first
    .withColumn("tail", F.expr("slice(V2_path, 2, size(V2_path)-1)"))
    # pair up head[i], tail[i]
    .withColumn("pairs", F.arrays_zip("head", "tail"))
    # one row per (src, dst)
    .select(F.explode("pairs").alias("e"))
    .select(F.col("e.head").alias("src"), F.col("e.tail").alias("dst"))
)

df_graph_V2_arc_resimplify_r1.printSchema()

root
 |-- src: integer (nullable = true)
 |-- dst: integer (nullable = true)



In [28]:
# drop duplicates
df_graph_V2_arc_resimplify_r1 = df_graph_V2_arc_resimplify_r1.dropDuplicates(["src", "dst"])
df_graph_V2_arc_resimplify_r1.printSchema()

root
 |-- src: integer (nullable = true)
 |-- dst: integer (nullable = true)



In [29]:
t0 = time.time()

file_df_graph_V2_arc_resimplify_r1 = directory + '/' + tin_filename + '_ET_arc_from_r' + already_resimplify_idx + '_merge_r' + round_idx + '.parquet'
# file_df_graph_V2_arc_resimplify_r1 = directory + '/' + tin_filename + '_ET_arc_from_r1_merge_r1' + '.parquet'
df_graph_V2_arc_resimplify_r1.write.parquet(file_df_graph_V2_arc_resimplify_r1) 

t1 = time.time()
print("time cost:", t1-t0)

time cost: 11.241306066513062


In [30]:
file_df_graph_V2_node = directory + '/' + tin_filename + '_ET_node' + '.parquet'
df_graph_V2_node = spark.read.parquet(file_df_graph_V2_node)
df_graph_V2_node.printSchema()

root
 |-- id: integer (nullable = true)
 |-- tri: array (nullable = true)
 |    |-- element: integer (containsNull = true)



#### construct a Graph_ET

In [31]:
graph_ET = GraphFrame(df_graph_V2_node, df_graph_V2_arc_resimplify_r1)

In [32]:
# set a checkpoint directory to improve performance
# Checkpointing regularly helps recover from failures, clean shuffle files, shorten the
# lineage of the computation graph, and reduce the complexity of plan optimization.

spark.sparkContext.setCheckpointDir('checkpoints')

In [33]:
t_con_0 = time.time()
# result_con consists of two columns, vertex id, component
result_con_ET = graph_ET.connectedComponents()
result_con_ET.printSchema()

t_con_1 = time.time()
t_con = t_con_1 - t_con_0
print("time cost of connected components:",t_con)

root
 |-- id: integer (nullable = true)
 |-- tri: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- component: long (nullable = true)

time cost of connected components: 92.82620787620544


#### distinct number of components

In [34]:
result_con_ET_gpy = (
    result_con_ET
    .groupBy("component")
    .agg(
        F.collect_list(F.struct("id", "tri")).alias("items")   # keep id-tri paired
        # or use collect_set(...) if duplicates are possible and you want unique (id,tri)
    )
)

result_con_ET_gpy_filtered = result_con_ET_gpy.filter(F.size("items") > 1)
num_comp_distinct_con_ET = result_con_ET_gpy_filtered.count()
print("The number of distinct components:", num_comp_distinct_con_ET)

result_con_ET_resimplify_r1_final = (
    result_con_ET_gpy_filtered
    .select(F.explode("items").alias("x"), F.col("component"))
    .select(F.col("x.id").alias("id"), F.col("x.tri").alias("tri"), F.col("component"))
)

result_con_ET_resimplify_r1_final.printSchema()

The number of distinct components: 137406
root
 |-- id: integer (nullable = true)
 |-- tri: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- component: long (nullable = true)



In [35]:
# _df_crit_V_update_no_threshold is the critical vertices after update but without resimplification
if round_idx == '1':
    file_df_crit_T = directory + '/' + tin_filename + '_df_crit_T_update_resimplify_r' + already_resimplify_idx + '.parquet'
else:
    file_df_crit_T = directory + '/' + tin_filename + '_df_crit_T_update_from_r' + already_resimplify_idx + '_merge_r' + str(int(round_idx)-1) + '.parquet'

# file_df_crit_T = directory + '/' + tin_filename + '_df_crit_T_update_resimplify_r1' + '.parquet'
df_crit_T = spark.read.parquet(file_df_crit_T)
df_crit_T.printSchema()

num_df_crit_T = df_crit_T.count()
print("The number of critical triangles in df_crit_T_update:", num_df_crit_T)

root
 |-- Max_tri: array (nullable = true)
 |    |-- element: integer (containsNull = true)

The number of critical triangles in df_crit_T_update: 144975


In [36]:
t_0 = time.time()

file_result_con_ET_resimplify_r1_final = directory + '/' + tin_filename + '_result_con_ET_from_r' + already_resimplify_idx + '_merge_r' + round_idx + '.parquet'

# file_result_con_ET_resimplify_r1_final = directory + '/' + tin_filename + '_result_con_ET_from_r1_merge_r1' + '.parquet'
result_con_ET_resimplify_r1_final.write.parquet(file_result_con_ET_resimplify_r1_final)

t_1 = time.time()
print("time cost of connected components:",t_1 - t_0)

time cost of connected components: 7.080708265304565


In [37]:
t_whole_program_1 = time.time()
print(f"Total time cost (s): {(t_whole_program_1 - t_whole_program_0)}")
print(f"Total time cost (min): {(t_whole_program_1 - t_whole_program_0) / 60}")

t_step2 = t_whole_program_1 - t_whole_program_0
print(f"Total time cost (min) for step 2: {t_step2 / 60}")

Total time cost (s): 272.2479770183563
Total time cost (min): 4.537466283639272
Total time cost (min) for step 2: 4.537466283639272


# Step 3. save potential simplex pairs to remove

### Extract critical vertices, edeges, and triangles

In [38]:
if round_idx == '1':
    file_df_crit_E = directory + '/' + tin_filename + '_df_crit_E_update_resimplify_r' + already_resimplify_idx + '.parquet'
else:
    file_df_crit_E = directory + '/' + tin_filename + '_df_crit_E_update_from_r' + already_resimplify_idx + '_merge_r' + str(int(round_idx)-1) + '.parquet'

# file_df_crit_E = directory + '/' + tin_filename + '_df_crit_E_update_resimplify_r1' + '.parquet'
df_crit_E = spark.read.parquet(file_df_crit_E)
df_crit_E.printSchema()

root
 |-- Ver: integer (nullable = true)
 |-- Saddle_edge: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [39]:
if round_idx == '1':
    file_df_crit_T = directory + '/' + tin_filename + '_df_crit_T_update_resimplify_r' + already_resimplify_idx + '.parquet'
else:
    file_df_crit_T = directory + '/' + tin_filename + '_df_crit_T_update_from_r' + already_resimplify_idx + '_merge_r' + str(int(round_idx)-1) + '.parquet'
    
#file_df_crit_T = directory + '/' + tin_filename + '_df_crit_T_update_resimplify_r1' + '.parquet'
df_crit_T = spark.read.parquet(file_df_crit_T)
df_crit_T.printSchema()

root
 |-- Max_tri: array (nullable = true)
 |    |-- element: integer (containsNull = true)



### Methods to build Graph_VE

#### get result_con of G_VE

In [40]:
file_result_con_VE = directory + '/' + tin_filename + '_result_con_VE_from_r' + already_resimplify_idx + '_merge_r' + round_idx + '.parquet'

# file_result_con_VE = directory + '/' + tin_filename + '_result_con_VE_from_r1_merge_r1' + '.parquet'
result_con_VE = spark.read.parquet(file_result_con_VE)
result_con_VE.printSchema()

root
 |-- id: integer (nullable = true)
 |-- component: long (nullable = true)



##### 1) get the boundary vertices of saddles

In [41]:
df_crit_E_pts = (df_crit_E.select(col("Saddle_edge"), explode(col("Saddle_edge")).alias("Saddle_bound_pt")))
df_crit_E_pts.printSchema()

root
 |-- Saddle_edge: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- Saddle_bound_pt: integer (nullable = true)



##### 2) inner join result_con_VE with the boundary vertices of saddles to identify the saddles that belong to the same component; typically a saddle belongs to 2 components in Graph_VE

In [42]:
# broadcast left join connected components with saddles
result_con_saddle_VE = result_con_VE.join(df_crit_E_pts,result_con_VE.id==df_crit_E_pts.Saddle_bound_pt, "inner")
result_con_saddle_VE.printSchema()

root
 |-- id: integer (nullable = true)
 |-- component: long (nullable = true)
 |-- Saddle_edge: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- Saddle_bound_pt: integer (nullable = true)



In [43]:
saddles_per_con_VE = result_con_saddle_VE.groupBy('component').agg(collect_list('Saddle_edge').alias('multi_Saddles'), collect_list('Saddle_bound_pt').alias('multi_Saddles_pts'))
saddles_per_con_VE.printSchema()

root
 |-- component: long (nullable = true)
 |-- multi_Saddles: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- multi_Saddles_pts: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [44]:
t0 = time.time()

file_saddles_per_con_VE = directory + '/' + tin_filename + '_saddles_per_con_VE_from_r' + already_resimplify_idx + '_merge_r' + round_idx + '.parquet'

# file_saddles_per_con_VE = directory + '/' + tin_filename + '_saddles_per_con_VE_from_r1_merge_r1' + '.parquet'
saddles_per_con_VE.write.parquet(file_saddles_per_con_VE) 

t1 = time.time()
print("time cost:", t1-t0)

time cost: 2.326624870300293


### Methods to build Graph_ET

#### get result_con_ET

In [45]:
file_result_con_ET = directory + '/' + tin_filename + '_result_con_ET_from_r' + already_resimplify_idx + '_merge_r' + round_idx + '.parquet'

# file_result_con_ET = directory + '/' + tin_filename + '_result_con_ET_from_r1_merge_r1' + '.parquet'
result_con_ET = spark.read.parquet(file_result_con_ET)
result_con_ET.printSchema()

root
 |-- id: integer (nullable = true)
 |-- tri: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- component: long (nullable = true)



##### 1)  get the co-boundary triangles of saddles

In [46]:
if round_idx == '1':
    file_df_crit_E_Co_tris_origin = directory + '/' + tin_filename + '_df_crit_E_Co_tris_resimplify_r' + already_resimplify_idx + '.parquet'
else:
    file_df_crit_E_Co_tris_origin = directory + '/' + tin_filename + '_df_crit_E_Co_tris_from_r' + already_resimplify_idx + '_merge_r' + str(int(round_idx)-1) + '.parquet'

# file_df_crit_E_Co_tris_origin = directory + '/' + tin_filename + '_df_crit_E_Co_tris_resimplify_r1' + '.parquet'
df_crit_E_Co_tris_origin = spark.read.parquet(file_df_crit_E_Co_tris_origin)
df_crit_E_Co_tris_origin.printSchema()

root
 |-- Saddle_edge: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- Saddle_Co_t: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [47]:
df_crit_E_Co_tris = df_crit_E_Co_tris_origin.join(df_crit_E, df_crit_E_Co_tris_origin.Saddle_edge==df_crit_E.Saddle_edge, "inner")
df_crit_E_Co_tris = df_crit_E_Co_tris.select(df_crit_E_Co_tris_origin.Saddle_edge, df_crit_E_Co_tris_origin.Saddle_Co_t)
df_crit_E_Co_tris.printSchema()

root
 |-- Saddle_edge: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- Saddle_Co_t: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [48]:
t0 = time.time()

file_df_crit_E_Co_tris = directory + '/' + tin_filename + '_df_crit_E_Co_tris_from_r' + already_resimplify_idx + '_merge_r' + round_idx + '.parquet'

# file_df_crit_E_Co_tris = directory + '/' + tin_filename + '_df_crit_E_Co_tris_from_r1_merge_r1' + '.parquet'
df_crit_E_Co_tris.write.parquet(file_df_crit_E_Co_tris) 

t1 = time.time()
print("time cost:", t1-t0)

time cost: 1.4692487716674805


##### 2) inner join result_con_ET with the co-boundary triangles of saddles to identify the saddles that belong to the same component; typically a saddle belongs to 2 component in Graph_ET

In [49]:
# broadcast left join connected components with saddles
saddles_per_con_ET_init = result_con_ET.join(df_crit_E_Co_tris,result_con_ET.tri==df_crit_E_Co_tris.Saddle_Co_t, "inner")

saddles_per_con_ET = saddles_per_con_ET_init.groupBy('component').agg(collect_list('Saddle_edge').alias('multi_Saddles'), collect_list('id').alias('multi_Saddles_co_tri_id'))
saddles_per_con_ET.printSchema()

root
 |-- component: long (nullable = true)
 |-- multi_Saddles: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- multi_Saddles_co_tri_id: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [50]:
t0 = time.time()

file_saddles_per_con_ET = directory + '/' + tin_filename + '_saddles_per_con_ET_from_r' + already_resimplify_idx + '_merge_r' + round_idx + '.parquet'

# file_saddles_per_con_ET = directory + '/' + tin_filename + '_saddles_per_con_ET_from_r1_merge_r1' + '.parquet'
saddles_per_con_ET.write.parquet(file_saddles_per_con_ET) 

t1 = time.time()
print("time cost:", t1-t0)

time cost: 2.9175703525543213


##### 3) inner join result_con with critical triangles to identify the component id of critical triangle 

In [51]:
df_crit_T = df_crit_T.select("Max_tri")

df_crit_T_con = df_crit_T.join(result_con_ET, df_crit_T.Max_tri==result_con_ET.tri, "left")
df_crit_T_con = df_crit_T_con.select("id", "component", "Max_tri")
df_crit_T_con.printSchema()

root
 |-- id: integer (nullable = true)
 |-- component: long (nullable = true)
 |-- Max_tri: array (nullable = true)
 |    |-- element: integer (containsNull = true)



##### 4) join saddles_per_con_ET with df_crit_T_con to get the corresponding critical triangle of a component ID

In [52]:
result_con_saddle_ET = saddles_per_con_ET.join(df_crit_T_con, saddles_per_con_ET.component==df_crit_T_con.component, "left")
result_con_saddle_ET = result_con_saddle_ET.select(saddles_per_con_ET.component, "Max_tri", "multi_Saddles")
result_con_saddle_ET.printSchema()

root
 |-- component: long (nullable = true)
 |-- Max_tri: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- multi_Saddles: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)



### 2. get the minimal saddle-maximum pairs belonging to the same component in G_ET

In [53]:
# get the potential saddle to be simplified within each component
# the potential saddle-maximum pair should have the minimal persistence value among all saddle-maximum pairs
def get_mini_saddle_ET(Max_tri, multi_Saddles):
    # Max_tri: the critical triangle in this component
    # multi_Saddles: the list of saddles in the same componnt as Max_tri
    if len(multi_Saddles) > 0:
        # initialize the potential minimal saddle
        mini_saddle = multi_Saddles[0]
        mini_saddle_persist = mini_saddle[0] # mini_saddle is a pair of vertex, and mini_saddle[0] > mini_saddle[1]
        
        for i in range(1, len(multi_Saddles)):
            if multi_Saddles[i][0] > mini_saddle_persist: # multi_Saddles[i][0] is the persist value of multi_Saddles[i]
                mini_saddle = multi_Saddles[i]
                mini_saddle_persist = mini_saddle[0]                
                
        return mini_saddle
    else:
        return None

get_mini_saddle_ET_udf = udf(get_mini_saddle_ET, ArrayType(IntegerType()))

result_con_saddle_ET_mini = result_con_saddle_ET.withColumn("mini_saddle", get_mini_saddle_ET_udf(result_con_saddle_ET.component, result_con_saddle_ET.multi_Saddles))
result_con_saddle_ET_mini.printSchema()

root
 |-- component: long (nullable = true)
 |-- Max_tri: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- multi_Saddles: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- mini_saddle: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [54]:
t0 = time.time()

file_result_con_saddle_ET_mini = directory + '/' + tin_filename + '_result_con_saddle_ET_mini_from_r' + already_resimplify_idx + '_merge_r' + round_idx + '.parquet'

# file_result_con_saddle_ET_mini = directory + '/' + tin_filename + '_result_con_saddle_ET_mini_from_r1_merge_r1' + '.parquet'
result_con_saddle_ET_mini.write.parquet(file_result_con_saddle_ET_mini) 

t1 = time.time()
print("time cost:", t1-t0)

time cost: 4.5929529666900635


### 3. make each saddle as a center, extract the minima and maxima connected to this saddle

In [55]:
con_saddle_1V1_ET = result_con_saddle_ET.select(col("component"), col("Max_tri"), explode(result_con_saddle_ET.multi_Saddles).alias("saddle_single_ET"))

Con_OF_saddle_ET = con_saddle_1V1_ET.groupby('saddle_single_ET').agg(collect_list('component').alias('multi_components_ET'), collect_list('Max_tri').alias('multi_Max_tri'))
Con_OF_saddle_ET.printSchema()

root
 |-- saddle_single_ET: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- multi_components_ET: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- multi_Max_tri: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)



### 4.2. join the component of maxima and the component of saddle to remove maxima-saddle pairs

In [56]:
DF_maxima_saddle_init = result_con_saddle_ET_mini.join(Con_OF_saddle_ET, result_con_saddle_ET_mini.mini_saddle==Con_OF_saddle_ET.saddle_single_ET, "left")
DF_maxima_saddle_init = DF_maxima_saddle_init.select(Con_OF_saddle_ET.saddle_single_ET.alias("pot_saddle"), result_con_saddle_ET_mini.Max_tri.alias("pot_maximum"), result_con_saddle_ET_mini.component.alias("pot_maximum_id"), "multi_Max_tri")
DF_maxima_saddle_init.printSchema()

root
 |-- pot_saddle: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- pot_maximum: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- pot_maximum_id: long (nullable = true)
 |-- multi_Max_tri: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)



In [57]:
t0 = time.time()

def hdfs_path_exists(spark, path_str):
    jvm = spark._jvm
    hadoop_conf = spark._jsc.hadoopConfiguration()
    fs = jvm.org.apache.hadoop.fs.FileSystem.get(hadoop_conf)
    path = jvm.org.apache.hadoop.fs.Path(path_str)
    return fs.exists(path)


# get the elevation values of each vertex
file_df_ver_order_ele_sort = directory + '/' + tin_filename + '_filtra_pts_sort' + '.parquet'
if hdfs_path_exists(spark, file_df_ver_order_ele_sort):
    df_ver_order_read = spark.read.parquet(file_df_ver_order_ele_sort)
    ver_dict = {row['self_order']: row['ele'] for row in df_ver_order_read.collect()}
else:
    ver_dict = {row['self_order']: row['ele'] for row in df_ver_order.collect()}

# Broadcast it (this is handled differently than a standard closure)
sc = spark.sparkContext
broadcast_ver = sc.broadcast(ver_dict)
# Access the dictionary via .value
lookup = broadcast_ver.value

t1 = time.time()
print("time cost:", t1-t0)

time cost: 75.99241161346436


In [58]:
t0 = time.time()

if simplify_with_order == '1' or simplify_with_order == 'yes' or simplify_with_order == 'y':
    # check if the maxima-saddle pair could be contracted
    def contract_ET(pot_saddle, pot_maximum, multi_Max_tri):
        # pot_saddle: the center saddle, which is the saddle to be contracted in the maxima-saddle pair
        # pot_maximum: the critical triangle to be contracted in the maxima-saddle pair
        # multi_Max_tri: the critical triangles that are connected with the pot_saddle
        if pot_saddle == None or pot_maximum == None:
            return
        
        persist_value = pot_maximum[0] - pot_saddle[0]
#         persist_value_thre = 1000
#         if persist_value < persist_value_thre:
#             less_than_thre = True
#         else:
#             less_than_thre = False
            
        less_than_thre = True
        smallest_max_saddle = True
                    
        if len(multi_Max_tri) > 0:
            for crit_tri in multi_Max_tri:
                if crit_tri[0] - pot_saddle[0] < persist_value:
                    smallest_max_saddle = False
            
        contract = smallest_max_saddle and less_than_thre
        return contract
else:
    print("Simplification according to elevation!")
    # get the elevation values of each vertex
#     file_df_ver_order_ele_sort = directory + '/' + tin_filename + '_filtra_pts_sort' + '.parquet'
#     if hdfs_path_exists(spark, file_df_ver_order_ele_sort):
#         df_ver_order_read = spark.read.parquet(file_df_ver_order_ele_sort)
#         df_ver_order_col = df_ver_order_read.collect()
#     else:
#         df_ver_order = df_ver_order.sort(col('self_order'), ascending=True)
#         df_ver_order_col = df_ver_order.collect()
    
    # check if the maxima-saddle pair could be contracted
    def contract_ET(pot_saddle, pot_maximum, multi_Max_tri):
        # pot_saddle: the center saddle, which is the saddle to be contracted in the maxima-saddle pair
        # pot_maximum: the critical triangle to be contracted in the maxima-saddle pair
        # multi_Max_tri: the critical triangles that are connected with the pot_saddle
        if pot_saddle == None or pot_maximum == None:
            return
        
        larger_than_HA = True
        # if df_ver_order_col[pot_maximum[0]]['ele'] < 0.6:
        if lookup.get(pot_maximum[0]) < 0.6:
            larger_than_HA = False
            
        # persist_value = abs(df_ver_order_col[pot_maximum[0]]['ele'] - df_ver_order_col[pot_saddle[0]]['ele'])
        persist_value = abs(lookup.get(pot_maximum[0]) - lookup.get(pot_saddle[0]))
        persist_value_thre = float(porsist_thre)
#         if persist_value < persist_value_thre:
#             less_than_thre = True
#         else:
#             less_than_thre = False
            
        less_than_thre = True
        smallest_max_saddle = True
                    
        if len(multi_Max_tri) > 0:
            for crit_tri in multi_Max_tri:
                if abs(lookup.get(crit_tri[0]) - lookup.get(pot_saddle[0])) < persist_value:
                # if abs(df_ver_order_col[crit_tri[0]]['ele'] - df_ver_order_col[pot_saddle[0]]['ele']) < persist_value:
                    smallest_max_saddle = False
            
        contract = larger_than_HA and smallest_max_saddle and less_than_thre
        return contract

contract_ET_udf = udf(contract_ET, BooleanType())

DF_maxima_saddle = DF_maxima_saddle_init.withColumn("contract_ET_or_not", contract_ET_udf(DF_maxima_saddle_init.pot_saddle, DF_maxima_saddle_init.pot_maximum, DF_maxima_saddle_init.multi_Max_tri))
DF_maxima_saddle.printSchema()

t1 = time.time()
print("time cost:", t1-t0)

Simplification according to elevation!
root
 |-- pot_saddle: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- pot_maximum: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- pot_maximum_id: long (nullable = true)
 |-- multi_Max_tri: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- contract_ET_or_not: boolean (nullable = true)

time cost: 31.604790449142456


In [59]:
t0 = time.time()

file_DF_maxima_saddle = directory + '/' + tin_filename + '_DF_maxima_saddle_from_r' + already_resimplify_idx + '_merge_r' + round_idx + '.parquet'

# file_DF_maxima_saddle = directory + '/' + tin_filename + '_DF_maxima_saddle_from_r1_merge_r1' + '.parquet'
DF_maxima_saddle.write.mode("overwrite").parquet(file_DF_maxima_saddle)

t1 = time.time()
print("time cost:", t1-t0)

time cost: 19.28296160697937


In [60]:
t_whole_program_2 = time.time()
print(f"Total time cost (s): {(t_whole_program_2 - t_whole_program_1)}")
print(f"Total time cost (min): {(t_whole_program_2 - t_whole_program_1) / 60}")

t_step3 = t_whole_program_2 - t_whole_program_1
print(f"Total time cost (min) for step 3: {t_step3 / 60}")

Total time cost (s): 139.71964931488037
Total time cost (min): 2.328660821914673
Total time cost (min) for step 3: 2.328660821914673


# Step 4. save V1- and V2- paths

### 1. Methods to build Graph_VE

In [61]:
file_df_graph_V1_arc = directory + '/' + tin_filename + '_VE_arc_from_r' + already_resimplify_idx + '_merge_r' + round_idx + '.parquet'

# file_df_graph_V1_arc = directory + '/' + tin_filename + '_VE_arc_from_r1_merge_r1' + '.parquet'
df_graph_V1_arc = spark.read.parquet(file_df_graph_V1_arc)
df_graph_V1_arc.printSchema()

root
 |-- src: integer (nullable = true)
 |-- dst: integer (nullable = true)



#### get result_con of G_VE

In [62]:
file_result_con_VE = directory + '/' + tin_filename + '_result_con_VE_from_r' + already_resimplify_idx + '_merge_r' + round_idx + '.parquet'

# file_result_con_VE = directory + '/' + tin_filename + '_result_con_VE_from_r1_merge_r1' + '.parquet'
result_con_VE = spark.read.parquet(file_result_con_VE)
result_con_VE.printSchema() # have checked that each id is identical in result_con_VE

root
 |-- id: integer (nullable = true)
 |-- component: long (nullable = true)



In [63]:
file_saddles_per_con_VE = directory + '/' + tin_filename + '_saddles_per_con_VE_from_r' + already_resimplify_idx + '_merge_r' + round_idx + '.parquet'

# file_saddles_per_con_VE = directory + '/' + tin_filename + '_saddles_per_con_VE_from_r1_merge_r1' + '.parquet'
saddles_per_con_VE = spark.read.parquet(file_saddles_per_con_VE)
saddles_per_con_VE.printSchema()

root
 |-- component: long (nullable = true)
 |-- multi_Saddles: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- multi_Saddles_pts: array (nullable = true)
 |    |-- element: integer (containsNull = true)



### 2. get V1-paths

##### get the extreme vertices of saddles that belong to the same component

In [64]:
saddles_Pts_per_con_VE = saddles_per_con_VE.select("component", "multi_Saddles_pts")
saddles_Pts_per_con_VE.printSchema()

root
 |-- component: long (nullable = true)
 |-- multi_Saddles_pts: array (nullable = true)
 |    |-- element: integer (containsNull = true)



##### get the directed arcs (VE pairs) that belong to the same component

In [65]:
# left join df_graph_V1_arc with result_con to identify which component that each arc belongs to
df_G_V1_arc_component = df_graph_V1_arc.join(result_con_VE,df_graph_V1_arc.src==result_con_VE.id, "left").drop('id')
df_G_V1_arc_component.printSchema()

root
 |-- src: integer (nullable = true)
 |-- dst: integer (nullable = true)
 |-- component: long (nullable = true)



##### store each component in the same row of a DataFrame

In [66]:
# convert each arc to a dictionary
df_G_V1_arc_component_dict = df_G_V1_arc_component.withColumn("arcs", create_map(col('src'), col('dst'))).drop('src','dst')

# combine dictionaries belonging to an identical component to one list
df_G_V1_arc_component_dict_gpy = df_G_V1_arc_component_dict.groupBy('component').agg(collect_list('arcs').alias('subgraphs'))

# left join df_G_V1_arc_component_dict_gpy with unique_SdlPts_per_con to obtain SdlPts belonging to each component
df_G_V1_arc_component_dict_gpy = df_G_V1_arc_component_dict_gpy.join(saddles_Pts_per_con_VE,df_G_V1_arc_component_dict_gpy.component==saddles_Pts_per_con_VE.component, "left").select(df_G_V1_arc_component_dict_gpy.component, 'multi_Saddles_pts', 'subgraphs')
df_G_V1_arc_component_dict_gpy.printSchema()

root
 |-- component: long (nullable = true)
 |-- multi_Saddles_pts: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- subgraphs: array (nullable = true)
 |    |-- element: map (containsNull = true)
 |    |    |-- key: integer
 |    |    |-- value: integer (valueContainsNull = true)



In [67]:
import collections
def bfs_12152025(multi_Saddles_pts, subgraphs):
    '''
    multi_Saddles_pts: an array of string or an array of integers
    subgraphs: an array of dictionary, where each key is a string (or integer) and the corresponding value is also a string (or integer)
    '''
    if not multi_Saddles_pts:
        return None
    # Build the adjacency list from the list of edge dictionaries
    adj_list = collections.defaultdict()
    for edge_dict in subgraphs:
        # Assuming each dictionary has only one key-value pair representing an edge
        for source, destination in edge_dict.items():
            adj_list[source] = destination

    V1_paths = []
    for i in range(len(multi_Saddles_pts)):
        node = multi_Saddles_pts[i]
        V1_path_temp = []
        V1_path_temp_set = set()
        
        while node != -1 and node not in V1_path_temp_set:
            V1_path_temp.append(node)
            V1_path_temp_set.add(node)
            next_node = adj_list.get(node, -1)
            node = next_node
        
        V1_paths.append(V1_path_temp)
    
    return V1_paths

bfs_df_udf = udf(bfs_12152025, ArrayType(ArrayType(IntegerType())))

In [68]:
# Use mapPartitions to apply the bfs function to each partition of the RDD
# Each partition corresponds to a single graph that is processed by a single executor
t_V1_0 = time.time()

df_V1_result_VE_12152025 = df_G_V1_arc_component_dict_gpy.withColumn("V1_paths", bfs_df_udf(df_G_V1_arc_component_dict_gpy.multi_Saddles_pts, df_G_V1_arc_component_dict_gpy.subgraphs))
df_V1_result_VE_12152025.printSchema()
# num_df_V1_result_VE = df_V1_result_VE.count()

t_V1_1 = time.time()
print("Time cost to extract V1-paths of Graph_VE:", t_V1_1 - t_V1_0)
# print("The number of V1-paths in Graph_VE:", num_df_V1_result_VE)

root
 |-- component: long (nullable = true)
 |-- multi_Saddles_pts: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- subgraphs: array (nullable = true)
 |    |-- element: map (containsNull = true)
 |    |    |-- key: integer
 |    |    |-- value: integer (valueContainsNull = true)
 |-- V1_paths: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)

Time cost to extract V1-paths of Graph_VE: 0.023035526275634766


In [69]:
t0 = time.time()

file_df_V1_result_VE_12152025 = directory + '/' + tin_filename + '_df_V1_result_VE_from_r' + already_resimplify_idx + '_merge_r' + round_idx + '.parquet'

# file_df_V1_result_VE_12152025 = directory + '/' + tin_filename + '_df_V1_result_VE_from_r1_merge_r1' + '.parquet'
df_V1_result_VE_12152025.write.parquet(file_df_V1_result_VE_12152025) 

t1 = time.time()
print("time cost:", t1-t0)

time cost: 3.7351489067077637


### 3. Methods to build Graph_ET

In [70]:
file_df_graph_V2_edge_final = directory + '/' + tin_filename + '_ET_arc_from_r' + already_resimplify_idx + '_merge_r' + round_idx + '.parquet'

# file_df_graph_V2_edge_final = directory + '/' + tin_filename + '_ET_arc_from_r1_merge_r1' + '.parquet'
df_graph_V2_edge_final = spark.read.parquet(file_df_graph_V2_edge_final)
df_graph_V2_edge_final.printSchema()

root
 |-- src: integer (nullable = true)
 |-- dst: integer (nullable = true)



#### get result_con_ET

In [71]:
file_result_con_ET = directory + '/' + tin_filename + '_result_con_ET_from_r' + already_resimplify_idx + '_merge_r' + round_idx + '.parquet'

# file_result_con_ET = directory + '/' + tin_filename + '_result_con_ET_from_r1_merge_r1' + '.parquet'
result_con_ET = spark.read.parquet(file_result_con_ET)
result_con_ET.printSchema()

root
 |-- id: integer (nullable = true)
 |-- tri: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- component: long (nullable = true)



In [72]:
file_saddles_per_con_ET = directory + '/' + tin_filename + '_saddles_per_con_ET_from_r' + already_resimplify_idx + '_merge_r' + round_idx + '.parquet'

# file_saddles_per_con_ET = directory + '/' + tin_filename + '_saddles_per_con_ET_from_r1_merge_r1' + '.parquet'
saddles_per_con_ET = spark.read.parquet(file_saddles_per_con_ET)
saddles_per_con_ET.printSchema()

root
 |-- component: long (nullable = true)
 |-- multi_Saddles: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- multi_Saddles_co_tri_id: array (nullable = true)
 |    |-- element: integer (containsNull = true)



### 4. get V2-paths

##### get the co-boundary triangles of saddles that belong to the same component

In [73]:
saddles_Tri_per_con_ET = saddles_per_con_ET.select("component", "multi_Saddles_co_tri_id")
saddles_Tri_per_con_ET.printSchema()

root
 |-- component: long (nullable = true)
 |-- multi_Saddles_co_tri_id: array (nullable = true)
 |    |-- element: integer (containsNull = true)



##### get the ET pairs (arcs of Graph_ET) that belong to the same component

In [74]:
# left join df_graph_V1_arc with result_con to identify which component that each arc belongs to
df_G_V2_arc_component = df_graph_V2_edge_final.join(result_con_ET,df_graph_V2_edge_final.src==result_con_ET.id, "left").drop('id', 'tri')
df_G_V2_arc_component.printSchema()

root
 |-- src: integer (nullable = true)
 |-- dst: integer (nullable = true)
 |-- component: long (nullable = true)



In [75]:
# convert each arc to a dictionary
df_G_V2_arc_component_dict = df_G_V2_arc_component.withColumn("arcs", create_map(col('src'), col('dst'))).drop('src','dst')

# combine dictionaries belonging to an identical component to one list
df_G_V2_arc_component_dict_gpy = df_G_V2_arc_component_dict.groupBy('component').agg(collect_list('arcs').alias('subgraphs'))

# left join df_G_V1_arc_component_dict_gpy with unique_SdlPts_per_con to obtain SdlPts belonging to each component
df_G_V2_arc_component_dict_gpy = df_G_V2_arc_component_dict_gpy.join(saddles_Tri_per_con_ET,df_G_V2_arc_component_dict_gpy.component==saddles_Tri_per_con_ET.component, "left").select(df_G_V2_arc_component_dict_gpy.component, 'multi_Saddles_co_tri_id', 'subgraphs')
df_G_V2_arc_component_dict_gpy.printSchema()

root
 |-- component: long (nullable = true)
 |-- multi_Saddles_co_tri_id: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- subgraphs: array (nullable = true)
 |    |-- element: map (containsNull = true)
 |    |    |-- key: integer
 |    |    |-- value: integer (valueContainsNull = true)



In [76]:
# Use mapPartitions to apply the bfs function to each partition of the RDD
# Each partition corresponds to a single graph that is processed by a single executor
t_V1_0 = time.time()

df_V2_result_ET = df_G_V2_arc_component_dict_gpy.withColumn("V2_paths", bfs_df_udf(df_G_V2_arc_component_dict_gpy.multi_Saddles_co_tri_id, df_G_V2_arc_component_dict_gpy.subgraphs))
df_V2_result_ET.printSchema()
# num_df_V2_result_ET = df_V2_result_ET.count()

t_V1_1 = time.time()
print("Time cost to extract V1-paths of Graph_VE:", t_V1_1 - t_V1_0)
# print("The number of V1-paths in Graph_VE:", num_df_V2_result_ET)

root
 |-- component: long (nullable = true)
 |-- multi_Saddles_co_tri_id: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- subgraphs: array (nullable = true)
 |    |-- element: map (containsNull = true)
 |    |    |-- key: integer
 |    |    |-- value: integer (valueContainsNull = true)
 |-- V2_paths: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)

Time cost to extract V1-paths of Graph_VE: 0.0054318904876708984


In [77]:
t0 = time.time()

file_df_V2_result_ET = directory + '/' + tin_filename + '_df_V2_result_ET_from_r' + already_resimplify_idx + '_merge_r' + round_idx + '.parquet'

# file_df_V2_result_ET = directory + '/' + tin_filename + '_df_V2_result_ET_from_r1_merge_r1' + '.parquet'
df_V2_result_ET.write.parquet(file_df_V2_result_ET) 

t1 = time.time()
print("time cost:", t1-t0)

time cost: 4.031663179397583


In [78]:
t_whole_program_3 = time.time()
print(f"Total time cost (s): {(t_whole_program_3 - t_whole_program_2)}")
print(f"Total time cost (min): {(t_whole_program_3 - t_whole_program_2) / 60}")

t_step4 = t_whole_program_3 - t_whole_program_2
print(f"Total time cost (min) for step 4: {t_step4 / 60}")

Total time cost (s): 9.490580320358276
Total time cost (min): 0.15817633867263795
Total time cost (min) for step 4: 0.15817633867263795


# Step 5. reverse and eliminate V1- and V2- paths

In [79]:
file_DF_maxima_saddle = directory + '/' + tin_filename + '_DF_maxima_saddle_from_r' + already_resimplify_idx + '_merge_r' + round_idx + '.parquet'

# file_DF_maxima_saddle = directory + '/' + tin_filename + '_DF_maxima_saddle_from_r1_merge_r1' + '.parquet'
DF_maxima_saddle = spark.read.parquet(file_DF_maxima_saddle)
DF_maxima_saddle.printSchema()

root
 |-- pot_saddle: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- pot_maximum: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- pot_maximum_id: long (nullable = true)
 |-- multi_Max_tri: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- contract_ET_or_not: boolean (nullable = true)



In [80]:
file_df_V1_result_VE = directory + '/' + tin_filename + '_df_V1_result_VE_from_r' + already_resimplify_idx + '_merge_r' + round_idx + '.parquet'

# file_df_V1_result_VE = directory + '/' + tin_filename + '_df_V1_result_VE_from_r1_merge_r1' + '.parquet'
df_V1_result_VE = spark.read.parquet(file_df_V1_result_VE)
df_V1_result_VE.printSchema()

root
 |-- component: long (nullable = true)
 |-- multi_Saddles_pts: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- subgraphs: array (nullable = true)
 |    |-- element: map (containsNull = true)
 |    |    |-- key: integer
 |    |    |-- value: integer (valueContainsNull = true)
 |-- V1_paths: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)



In [81]:
file_df_V2_result_ET = directory + '/' + tin_filename + '_df_V2_result_ET_from_r' + already_resimplify_idx + '_merge_r' + round_idx + '.parquet'

# file_df_V2_result_ET = directory + '/' + tin_filename + '_df_V2_result_ET_from_r1_merge_r1' + '.parquet'
df_V2_result_ET = spark.read.parquet(file_df_V2_result_ET)
df_V2_result_ET.printSchema()

root
 |-- component: long (nullable = true)
 |-- multi_Saddles_co_tri_id: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- subgraphs: array (nullable = true)
 |    |-- element: map (containsNull = true)
 |    |    |-- key: integer
 |    |    |-- value: integer (valueContainsNull = true)
 |-- V2_paths: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)



In [82]:
file_result_con_VE = directory + '/' + tin_filename + '_result_con_VE_from_r' + already_resimplify_idx + '_merge_r' + round_idx + '.parquet'

# file_result_con_VE = directory + '/' + tin_filename + '_result_con_VE_from_r1_merge_r1' + '.parquet'
result_con_VE = spark.read.parquet(file_result_con_VE)
result_con_VE.printSchema()

root
 |-- id: integer (nullable = true)
 |-- component: long (nullable = true)



In [83]:
file_result_con_ET = directory + '/' + tin_filename + '_result_con_ET_from_r' + already_resimplify_idx + '_merge_r' + round_idx + '.parquet'

# file_result_con_ET = directory + '/' + tin_filename + '_result_con_ET_from_r1_merge_r1' + '.parquet'
result_con_ET = spark.read.parquet(file_result_con_ET)
result_con_ET.printSchema()

root
 |-- id: integer (nullable = true)
 |-- tri: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- component: long (nullable = true)



In [84]:
file_df_crit_E_Co_tris = directory + '/' + tin_filename + '_df_crit_E_Co_tris_from_r' + already_resimplify_idx + '_merge_r' + round_idx + '.parquet'

# file_df_crit_E_Co_tris = directory + '/' + tin_filename + '_df_crit_E_Co_tris_from_r1_merge_r1' + '.parquet'
df_crit_E_Co_tris = spark.read.parquet(file_df_crit_E_Co_tris)
df_crit_E_Co_tris.printSchema()

root
 |-- Saddle_edge: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- Saddle_Co_t: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [85]:
# broadcast left join connected components with saddles
saddles_per_con_ET_init = result_con_ET.join(df_crit_E_Co_tris,result_con_ET.tri==df_crit_E_Co_tris.Saddle_Co_t, "inner")

saddles_per_con_ET = saddles_per_con_ET_init.groupBy('component').agg(collect_list('Saddle_edge').alias('multi_Saddles'), collect_list('id').alias('multi_Saddles_co_tri_id'))
saddles_per_con_ET.printSchema()

root
 |-- component: long (nullable = true)
 |-- multi_Saddles: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- multi_Saddles_co_tri_id: array (nullable = true)
 |    |-- element: integer (containsNull = true)



### update V2-paths

#### reverse part of V2-paths

In [86]:
DF_maxima_saddle_True = DF_maxima_saddle.filter(DF_maxima_saddle.contract_ET_or_not == True)
DF_maxima_saddle_True.printSchema()

root
 |-- pot_saddle: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- pot_maximum: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- pot_maximum_id: long (nullable = true)
 |-- multi_Max_tri: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- contract_ET_or_not: boolean (nullable = true)



##### jion DF_saddle_maxima_True and df_V2_result_ET to reverse part of V2-paths

In [87]:
df_V2_result_ET_update_init = df_V2_result_ET.join(DF_maxima_saddle_True, df_V2_result_ET.component==DF_maxima_saddle_True.pot_maximum_id, "left")
df_V2_result_ET_update_init = df_V2_result_ET_update_init.select("component", "pot_maximum", "pot_saddle", "V2_paths", "multi_Saddles_co_tri_id")
df_V2_result_ET_update_init.printSchema()

root
 |-- component: long (nullable = true)
 |-- pot_maximum: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- pot_saddle: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- V2_paths: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- multi_Saddles_co_tri_id: array (nullable = true)
 |    |-- element: integer (containsNull = true)



##### get the triangle id of the co-boundary triangles of pot-saddles

In [88]:
saddles_per_con_ET_init_Co_t = saddles_per_con_ET_init.groupby("Saddle_edge").agg(collect_list('id').alias('two_Saddle_Co_t_id'))
saddles_per_con_ET_init_Co_t.printSchema()

root
 |-- Saddle_edge: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- two_Saddle_Co_t_id: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [89]:
df_V2_result_ET_update_init_2 = df_V2_result_ET_update_init.join(saddles_per_con_ET_init_Co_t, df_V2_result_ET_update_init.pot_saddle==saddles_per_con_ET_init_Co_t.Saddle_edge, "left")
df_V2_result_ET_update_init_2 = df_V2_result_ET_update_init_2.select("component", "pot_maximum", "pot_saddle", "two_Saddle_Co_t_id", "V2_paths", "multi_Saddles_co_tri_id")
df_V2_result_ET_update_init_2.printSchema()

root
 |-- component: long (nullable = true)
 |-- pot_maximum: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- pot_saddle: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- two_Saddle_Co_t_id: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- V2_paths: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- multi_Saddles_co_tri_id: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [90]:
# a function to reverse the Forman gradient (V1-paths)
def reverse_V2_ET(two_Saddle_Co_t_id, V2_paths):
    if two_Saddle_Co_t_id:
        index_to_reverse = -1
#         for i in range(len(V2_paths)):
#             if V2_paths[i][0] in two_Saddle_Co_t_id:
#                 index_to_reverse = i
#                 # get the co-boundary triangle this saddle and this co-boundary triangle is not contained in this component
#                 if V2_paths[i][0] == two_Saddle_Co_t_id[0]:
#                     pot_saddle_unique_co_t = two_Saddle_Co_t_id[1]
#                 else:
#                     pot_saddle_unique_co_t = two_Saddle_Co_t_id[0]
#                 break
        
        sad_Co_t0_connected_with_crit_t = False
        sad_Co_t1_connected_with_crit_t = False
        pot_saddle_unique_co_t = None
        for i in range(len(V2_paths)):
            if V2_paths[i][0] in two_Saddle_Co_t_id:
                index_to_reverse = i
#                 if len(two_Saddle_Co_t_id) < 2:
#                     break
                # get the co-boundary triangle this saddle and this co-boundary triangle is not contained in this component
                if len(two_Saddle_Co_t_id) > 1 and V2_paths[i][0] == two_Saddle_Co_t_id[0]:
                    pot_saddle_unique_co_t = two_Saddle_Co_t_id[1]
                    sad_Co_t0_connected_with_crit_t = True
                if len(two_Saddle_Co_t_id) > 1 and V2_paths[i][0] == two_Saddle_Co_t_id[1]:
                    pot_saddle_unique_co_t = two_Saddle_Co_t_id[0]
                    sad_Co_t1_connected_with_crit_t = True
        
#         if sad_Co_t0_connected_with_crit_t and sad_Co_t1_connected_with_crit_t:
#             # both of the two extreme vertices of a saddle are connected with crit_v
#             return None
                
        if index_to_reverse != -1:
           #  V2_paths[index_to_reverse].reverse()
            # concatenate the V-paths
            new_V2_paths = []
            set_reverse = set(V2_paths[index_to_reverse])
            for i in range(len(V2_paths)):
                if i != index_to_reverse:
                    # test if part of V2_paths[index_to_reverse] is already in V2_paths[i]
                    set1 = set(V2_paths[i])
                    common_tri_id = set1.intersection(set_reverse)
                    lst1 = V2_paths[i][0:(len(V2_paths[i]) - len(common_tri_id) + 1)]
                    lst2 = V2_paths[index_to_reverse][0:(len(V2_paths[index_to_reverse]) - len(common_tri_id))]
                    lst2.reverse()
                    if pot_saddle_unique_co_t and pot_saddle_unique_co_t != -1: # a boundary critical triangle if it is -1
                        lst2.append(pot_saddle_unique_co_t) # add the other end point of this saddle
                    new_V2_paths.append(lst1+lst2)
                    
            if len(new_V2_paths) == 0: # a special case when there is only one path in V2_paths
                V2_paths_origin = V2_paths[i]
                V2_paths_origin.reverse()
                if pot_saddle_unique_co_t and pot_saddle_unique_co_t != -1:
                    V2_paths_origin.append(pot_saddle_unique_co_t)
                    
                new_V2_paths.append(V2_paths_origin)
                
            return new_V2_paths
    else:
        return V2_paths

reverse_V2_ET_udf = udf(reverse_V2_ET, ArrayType(ArrayType(IntegerType())))

##### reverse V2_paths

In [91]:
df_V2_result_ET_update = df_V2_result_ET_update_init_2.withColumn("reversed_V2_paths", reverse_V2_ET_udf(df_V2_result_ET_update_init_2.two_Saddle_Co_t_id, df_V2_result_ET_update_init_2.V2_paths))
df_V2_result_ET_update.printSchema()

root
 |-- component: long (nullable = true)
 |-- pot_maximum: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- pot_saddle: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- two_Saddle_Co_t_id: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- V2_paths: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- multi_Saddles_co_tri_id: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- reversed_V2_paths: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)



In [92]:
t0 = time.time()

file_df_V2_result_ET_update = directory + '/' + tin_filename + '_df_V2_result_ET_update_from_r' + already_resimplify_idx + '_merge_r' + round_idx + '.parquet'

# file_df_V2_result_ET_update = directory + '/' + tin_filename + '_df_V2_result_ET_update_from_r1_merge_r1' + '.parquet'
df_V2_result_ET_update.write.mode("overwrite").parquet(file_df_V2_result_ET_update) 

t1 = time.time()
print("time cost:", t1-t0)

time cost: 7.954272031784058


### eliminate the involved V1-path

#### eliminate saddle-maximum and the involved V1-paths

In [93]:
DF_maxima_saddle_remove = DF_maxima_saddle_True.select(col("pot_saddle").alias("removed_saddle"), col("pot_maximum").alias("removed_crti_triangle"))
DF_maxima_saddle_remove.printSchema()

root
 |-- removed_saddle: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- removed_crti_triangle: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [94]:
t0 = time.time()

file_DF_maxima_saddle_remove = directory + '/' + tin_filename + '_DF_maxima_saddle_remove_from_r' + already_resimplify_idx + '_merge_r' + round_idx + '.parquet'

# file_DF_maxima_saddle_remove = directory + '/' + tin_filename + '_DF_maxima_saddle_remove_from_r1_merge_r1' + '.parquet'
DF_maxima_saddle_remove.write.mode("overwrite").parquet(file_DF_maxima_saddle_remove) 

t1 = time.time()
print("time cost:", t1-t0)

time cost: 0.37218332290649414


##### get the connected components in G_VE that each extreme vertex belongs to

In [95]:
DF_maxima_saddle_remove_End_pt_expl = DF_maxima_saddle_remove.select(explode("removed_saddle").alias("one_End_pt_id"))
DF_maxima_saddle_remove_End_pt_expl.printSchema()

root
 |-- one_End_pt_id: integer (nullable = true)



In [96]:
# get the component that one_End_pt_id belongs to
DF_SadMax_RM_End_pt_expl_com = DF_maxima_saddle_remove_End_pt_expl.join(result_con_VE, DF_maxima_saddle_remove_End_pt_expl.one_End_pt_id==result_con_VE.id, "left")
DF_SadMax_RM_End_pt_expl_com = DF_SadMax_RM_End_pt_expl_com.select("one_End_pt_id", col("component").alias("one_End_pt_id_component"))
DF_SadMax_RM_End_pt_expl_com.printSchema()

root
 |-- one_End_pt_id: integer (nullable = true)
 |-- one_End_pt_id_component: long (nullable = true)



In [97]:
DF_SadMax_RM_End_pt_expl_com_gp = DF_SadMax_RM_End_pt_expl_com.groupby("one_End_pt_id_component").agg(collect_list('one_End_pt_id').alias('one_End_pt_ids'))
DF_SadMax_RM_End_pt_expl_com_gp.printSchema()

root
 |-- one_End_pt_id_component: long (nullable = true)
 |-- one_End_pt_ids: array (nullable = true)
 |    |-- element: integer (containsNull = true)



##### eliminate the involved V1-paths

In [98]:
df_V1_result_VE_update_rm_init = df_V1_result_VE.select("component","V1_paths")

df_V1_result_VE_update_rm = df_V1_result_VE_update_rm_init.join(DF_SadMax_RM_End_pt_expl_com_gp, df_V1_result_VE_update_rm_init.component==DF_SadMax_RM_End_pt_expl_com_gp.one_End_pt_id_component, "left")
df_V1_result_VE_update_rm = df_V1_result_VE_update_rm.select("component","V1_paths", "one_End_pt_ids")
df_V1_result_VE_update_rm.printSchema()

root
 |-- component: long (nullable = true)
 |-- V1_paths: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- one_End_pt_ids: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [99]:
# a function to eliminate the involved V1-paths
def elim_V1_paths(reversed_V1_paths, one_End_pt_ids):
    if one_End_pt_ids and reversed_V1_paths: # this V1-path should be eliminated
        # for each sublist (the V2-path) in reversed_V2_paths, if its first element is not contained in one_Co_t_ids, we eliminate it
        filtered_V1_paths = list(filter(lambda sublist: sublist and sublist[0] not in one_End_pt_ids, reversed_V1_paths))
        return filtered_V1_paths
    else: # return the original V2-paths
        return reversed_V1_paths
    
elim_V1_paths_udf = udf(elim_V1_paths, ArrayType(ArrayType(IntegerType())))

In [100]:
df_V1_result_VE_update_final = df_V1_result_VE_update_rm.withColumn("final_V1_paths", elim_V1_paths_udf(df_V1_result_VE_update_rm.V1_paths, df_V1_result_VE_update_rm.one_End_pt_ids))
df_V1_result_VE_update_final = df_V1_result_VE_update_final.select("component", "final_V1_paths")
df_V1_result_VE_update_final.printSchema()

root
 |-- component: long (nullable = true)
 |-- final_V1_paths: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)



In [101]:
t0 = time.time()

file_df_V1_result_VE_update_final = directory + '/' + tin_filename + '_df_V1_result_VE_update_eliminate_from_r' + already_resimplify_idx + '_merge_r' + round_idx + '.parquet'

# file_df_V1_result_VE_update_final = directory + '/' + tin_filename + '_df_V1_result_VE_update_eliminate_from_r1_merge_r1' + '.parquet'
df_V1_result_VE_update_final.write.mode("overwrite").parquet(file_df_V1_result_VE_update_final) 

t1 = time.time()
print("time cost:", t1-t0)

time cost: 1.617922306060791


#### Update critical vertices, critical edges, and critical triangles

In [102]:
# _df_crit_V_update_no_threshold is the critical vertices after update but without resimplification
if round_idx == '1':
    file_df_crit_V = directory + '/' + tin_filename + '_df_crit_V_update_resimplify_r' + already_resimplify_idx + '.parquet'
else:
    file_df_crit_V = directory + '/' + tin_filename + '_df_crit_V_update_from_r' + already_resimplify_idx + '_merge_r' + str(int(round_idx)-1) + '.parquet'
    
# file_df_crit_V = directory + '/' + tin_filename + '_df_crit_V_update_resimplify_r1' + '.parquet'
df_crit_V = spark.read.parquet(file_df_crit_V)
df_crit_V.printSchema()

root
 |-- Ver: integer (nullable = true)
 |-- Min_ver: integer (nullable = true)



In [103]:
if round_idx == '1':
    file_df_crit_E = directory + '/' + tin_filename + '_df_crit_E_update_resimplify_r' + already_resimplify_idx + '.parquet'
else:
    file_df_crit_E = directory + '/' + tin_filename + '_df_crit_E_update_from_r' + already_resimplify_idx + '_merge_r' + str(int(round_idx)-1) + '.parquet'
    
# file_df_crit_E = directory + '/' + tin_filename + '_df_crit_E_update_resimplify_r1' + '.parquet'
df_crit_E = spark.read.parquet(file_df_crit_E)
df_crit_E.printSchema()

root
 |-- Ver: integer (nullable = true)
 |-- Saddle_edge: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [104]:
if round_idx == '1':
    file_df_crit_T = directory + '/' + tin_filename + '_df_crit_T_update_resimplify_r' + already_resimplify_idx + '.parquet'
else:
    file_df_crit_T = directory + '/' + tin_filename + '_df_crit_T_update_from_r' + already_resimplify_idx + '_merge_r' + str(int(round_idx)-1) + '.parquet'
    
# file_df_crit_T = directory + '/' + tin_filename + '_df_crit_T_update_resimplify_r1' + '.parquet'
df_crit_T = spark.read.parquet(file_df_crit_T)
df_crit_T.printSchema()

root
 |-- Max_tri: array (nullable = true)
 |    |-- element: integer (containsNull = true)



#### critical vertices will keep unchanged, which is still df_crit_V_update_resimplify_r5

In [105]:
t0 = time.time()

file_df_crit_V_update = directory + '/' + tin_filename + '_df_crit_V_update_from_r' + already_resimplify_idx + '_merge_r' + round_idx + '.parquet'

# file_df_crit_V_update = directory + '/' + tin_filename + '_df_crit_V_update_from_r1_merge_r1' + '.parquet'
df_crit_V.write.parquet(file_df_crit_V_update) 

t1 = time.time()
print("time cost:", t1-t0)

time cost: 0.4809761047363281


In [106]:
df_crit_T_update = df_crit_T.join(DF_maxima_saddle_remove, df_crit_T.Max_tri==DF_maxima_saddle_remove.removed_crti_triangle, "left")

df_crit_T_update = df_crit_T_update.filter(col("removed_crti_triangle").isNull())
df_crit_T_update = df_crit_T_update.select("Max_tri")

In [107]:
t0 = time.time()

file_df_crit_T_update = directory + '/' + tin_filename + '_df_crit_T_update_from_r' + already_resimplify_idx + '_merge_r' + round_idx + '.parquet'

# file_df_crit_T_update = directory + '/' + tin_filename + '_df_crit_T_update_from_r1_merge_r1' + '.parquet'
df_crit_T_update.write.mode("overwrite").parquet(file_df_crit_T_update) 

t1 = time.time()
print("time cost:", t1-t0)

time cost: 1.2124838829040527


In [108]:
DF_saddles_remove = DF_maxima_saddle_remove.select("removed_saddle")

DF_saddles_remove.printSchema()

root
 |-- removed_saddle: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [109]:
df_crit_E_update = df_crit_E.join(DF_saddles_remove, df_crit_E.Saddle_edge==DF_saddles_remove.removed_saddle, "left")

df_crit_E_update = df_crit_E_update.filter(col("removed_saddle").isNull())
df_crit_E_update = df_crit_E_update.select("Ver", "Saddle_edge")

In [110]:
t0 = time.time()

file_df_crit_E_update = directory + '/' + tin_filename + '_df_crit_E_update_from_r' + already_resimplify_idx + '_merge_r' + round_idx + '.parquet'

# file_df_crit_E_update = directory + '/' + tin_filename + '_df_crit_E_update_from_r1_merge_r1' + '.parquet'
df_crit_E_update.write.mode("overwrite").parquet(file_df_crit_E_update) 

t1 = time.time()
print("time cost:", t1-t0)

time cost: 1.5391268730163574


In [111]:
t_whole_program_4 = time.time()
print(f"Total time cost (s): {(t_whole_program_4 - t_whole_program_3)}")
print(f"Total time cost (min): {(t_whole_program_4 - t_whole_program_3) / 60}")

t_step5 = t_whole_program_4 - t_whole_program_3
print(f"Total time cost (min) for step 5: {t_step5 / 60}")

Total time cost (s): 15.607021808624268
Total time cost (min): 0.26011703014373777
Total time cost (min) for step 5: 0.26011703014373777


In [112]:
print(f"Total time cost (min) for step 2: {t_step2 / 60}")
print(f"Total time cost (min) for step 3: {t_step3 / 60}")
print(f"Total time cost (min) for step 4: {t_step4 / 60}")
print(f"Total time cost (min) for step 5: {t_step5 / 60}")

print(f"Total time cost (min) for the whole process: {(t_step2 + t_step3 + t_step4 + t_step5) / 60}")

Total time cost (min) for step 2: 4.537466283639272
Total time cost (min) for step 3: 2.328660821914673
Total time cost (min) for step 4: 0.15817633867263795
Total time cost (min) for step 5: 0.26011703014373777
Total time cost (min) for the whole process: 7.284420474370321
